# 🚢 تجارت‌یار — اجرا روی Colab (آپلود مستقیم ZIP)

این دفترچه **بدون نیاز به GitHub** سامانه را بالا می‌آورد؛ چون خروجی نهایی (`dist`) از قبل داخل فایل ZIP است.

> 🎬 **برای ارائه‌ی زنده** دفترچه‌ی `presentation_colab.ipynb` را باز کنید (تست سلامت + سناریوی ارائه).
> اگر GitHub از شبکه‌ی شما بسته است، همین دفترچه راه جایگزین ارائه است.

**مراحل — هر سلول را به ترتیب با `Ctrl+Enter` اجرا کنید:**
1. نصب Node.js (فقط در صورت نیاز)
2. آپلود فایل `Tejaratyarr.zip`
3. استخراج و آماده‌سازی
4. اجرای سرور در **حالت ارائه** (`SEED_DEMO=1`)
5. بررسی سلامت
6. 👀 **نمایش فوری در همین Colab** (بدون تونل)
7. دانلود cloudflared (فقط اگر لینک قابل ارسال می‌خواهید)
8. راه‌اندازی تونل
9. دریافت لینک عمومی

> برای **دیدن سریع برنامه** فقط تا سلول ۶ کافی است.

In [ ]:
%%bash
# نصب Node.js فقط در صورت نیاز (Colab معمولاً Node ۱۸ یا جدیدتر دارد)
MAJOR=$(node -v 2>/dev/null | sed 's/^v\([0-9]*\).*/\1/')
if [ -n "$MAJOR" ] && [ "$MAJOR" -ge 18 ]; then
  echo "node already present: $(node -v)"
else
  curl -fsSL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -o /tmp/node.txz \
    && tar -xJf /tmp/node.txz -C /usr/local --strip-components=1 \
    && echo "node installed: $(node -v)"
fi

In [ ]:
from google.colab import files

print("لطفاً فایل Tejaratyarr.zip را انتخاب و آپلود کنید:")
uploaded = files.upload()
print("آپلود شد:", list(uploaded.keys()))

In [ ]:
%%bash
Z=$(ls -t /content/*.zip 2>/dev/null | head -1); echo "zip file: $Z"
rm -rf /content/Tejaratyarr; unzip -oq "$Z" -d /content
echo "---"; ls -la /content/Tejaratyarr/dist/server.cjs

In [ ]:
%%bash
# اجرای سرور در حالت ارائه (SEED_DEMO=1 → کارتابل با داده‌ی نمونه پر می‌شود)
cd /content/Tejaratyarr || exit 1
pkill -f 'node dist/server.cjs' 2>/dev/null; sleep 1
SEED_DEMO=1 NODE_ENV=production setsid nohup node dist/server.cjs > /content/server.log 2>&1 < /dev/null &
sleep 1; echo "server starting... (log: /content/server.log)"

In [ ]:
%%bash
# بررسی سلامت + شمارش داده‌ی کارتابل (تا ۳۰ ثانیه تلاش می‌کند)
for i in $(seq 1 30); do
  H=$(curl -s http://localhost:3000/api/health)
  [ -n "$H" ] && break
  sleep 1
done
echo "health : $H"
curl -s http://localhost:3000/api/demo/state; echo
if [ -z "$H" ]; then echo "❌ سرور بالا نیامد — لاگ:"; tail -20 /content/server.log; fi

In [ ]:
from google.colab import output

# نمایش برنامه داخل همین دفترچه (بدون تونل و بدون انتظار)
output.serve_kernel_port_as_window(3000)

In [ ]:
%%bash
cd /content/Tejaratyarr || exit 1
if [ ! -x cloudflared ]; then
  curl -L --progress-bar -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
fi
chmod +x cloudflared && ./cloudflared --version

In [ ]:
import subprocess

# بستن هر نمونه‌ی قبلی cloudflared (اگر سلول را دوباره اجرا کنید)
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True, capture_output=True)

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:3000",
     "--no-autoupdate", "--logfile", "/content/cloudflared.log"],
    cwd="/content/Tejaratyarr",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, stdin=subprocess.DEVNULL,
    start_new_session=True,
)
print("cloudflared started (PID", p.pid, ") — برای گرفتن لینک به سلول بعد بروید.")

In [ ]:
import time, re

print("در انتظار لینک تونل (معمولاً ۱۰ تا ۳۰ ثانیه) ...")
url = None
for _ in range(60):
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                      open("/content/cloudflared.log", errors="ignore").read())
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

print()
if url:
    print("LINK:", url)
    print("(لینک موقت است و تا وقتی این Colab روشن بماند کار می‌کند)")
else:
    print("❌ لینک آماده نشد — همین سلول را دوباره اجرا کنید. آخرین خطوط لاگ:")
    try:
        print(open("/content/cloudflared.log", errors="ignore").read()[-2000:])
    except Exception as e:
        print(e)

## 📌 نکات

- **نمایش فوری:** سلول ۶ برنامه را داخل خود Colab نشان می‌دهد (برای ارائه روی صفحه‌ی خودتان کافی است).
- **لینک عمومی:** سلول ۹ یک لینک `trycloudflare` می‌دهد که تا وقتی Colab روشن است قابل ارسال است.
- **داده‌ی نمونه:** سلول ۴ سرور را با `SEED_DEMO=1` اجرا می‌کند تا کارتابل خالی نباشد (برچسب آن در هدر برنامه دیده می‌شود).
- بازنشانی داده‌ی نمونه وسط دمو: `curl -X POST http://localhost:3000/api/demo/seed`
- داده‌ها در Colab موقتی است؛ برای استفاده‌ی واقعی روی سرور خودتان اجرا کنید.
- فعال‌سازی هوش مصنوعی Gemini: قبل از سلول ۴، `GEMINI_API_KEY` را تنظیم کنید.
- اجرای محلی: `npm install` سپس `npm run dev` (پورت ۳۰۰۰).

## 🛠 رفع اشکال
- **آپلود نشد؟** سلول ۲ را دوباره اجرا کنید و مطمئن شوید فایل با پسوند zip انتخاب می‌شود.
- **سرور بالا نیامد؟** سلول ۵ را اجرا کنید؛ لاگ در `/content/server.log` است.
- **لینک چاپ نشد؟** سلول‌های ۸ و ۹ را دوباره اجرا کنید.